# Customer Churn Prediction

The goal of this project is to predict whether a customer will leave (churn) based on their demographic and service usage data.

By building and comparing multiple machine learning models, we aim to identify patterns that contribute to churn and evaluate which model performs best.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## Data Loading

In this step, we load the Telco Customer Churn dataset into a pandas DataFrame. This dataset contains customer demographic information, account details, and whether each customer has churned.

In [4]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Exploration

We examine the structure of the dataset to understand:
- The number of observations and features
- Data types of each column
- Whether any values are missing

This helps identify necessary cleaning steps before modeling.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [15]:
df.isnull().sum()

SeniorCitizen                            0
tenure                                   0
MonthlyCharges                           0
TotalCharges                             0
Churn                                    0
gender_Male                              0
Partner_Yes                              0
Dependents_Yes                           0
PhoneService_Yes                         0
MultipleLines_No phone service           0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_No internet service       0
OnlineSecurity_Yes                       0
OnlineBackup_No internet service         0
OnlineBackup_Yes                         0
DeviceProtection_No internet service     0
DeviceProtection_Yes                     0
TechSupport_No internet service          0
TechSupport_Yes                          0
StreamingTV_No internet service          0
StreamingTV_Yes                          0
StreamingMo

## Data Cleaning

To prepare the dataset for modeling:
- We remove the `customerID` column since it does not provide predictive value
- Convert `TotalCharges` to a numeric format (it is stored as text)
- Handle missing values by replacing them with the mean of the column

These steps ensure the dataset is consistent and usable for machine learning algorithms.

In [8]:
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df = df.fillna(df.mean(numeric_only=True))

## Feature Engineering

Machine learning models require numerical input, so we:
- Convert the target variable `Churn` into binary form (1 = churn, 0 = no churn)
- Apply one-hot encoding to categorical variables to convert them into numeric features

This step transforms the dataset into a format suitable for modeling.

In [9]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df = pd.get_dummies(df, drop_first=True)

## Train-Test Split

We split the dataset into training and testing sets:
- The training set is used to train the models
- The testing set is used to evaluate performance on unseen data

This helps ensure that our models generalize well and are not overfitting.

In [10]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

 ## Logistic Regression Model

We begin with Logistic Regression as a baseline model. This model estimates the probability that a customer will churn using a linear relationship between features and the target variable.

In [11]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.8211497515968772


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Decision Tree Model

Next, we train a Decision Tree model. This model captures non-linear relationships by splitting the data into decision rules based on feature values.

In [12]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

Decision Tree Accuracy: 0.7097232079489


## Random Forest Model

We train a Random Forest model, which is an ensemble of multiple decision trees. By combining many trees, this model typically improves prediction accuracy and reduces overfitting.

In [13]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

Random Forest Accuracy: 0.794180269694819


## Logistic Regression Feature Interpretation

To better understand the model, we examine the logistic regression coefficients. Positive coefficients increase the likelihood of churn, while negative coefficients decrease it.

In [17]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_[0]
})

coef_df = coef_df.sort_values(by="Coefficient", ascending=False)
coef_df.head(10)

,Feature,Coefficient
10,InternetService_Fiber optic,0.654757
26,PaperlessBilling_Yes,0.334754
28,PaymentMethod_Electronic check,0.324440
8,MultipleLines_No phone service,0.266120
23,StreamingMovies_Yes,0.231844
9,MultipleLines_Yes,0.223000
0,SeniorCitizen,0.159035
21,StreamingTV_Yes,0.128019
5,Partner_Yes,0.052369
2,MonthlyCharges,0.002880


In [18]:
coef_df.tail(10)

,Feature,Coefficient
18,TechSupport_No internet service,-0.099583
16,DeviceProtection_No internet service,-0.099583
14,OnlineBackup_No internet service,-0.099583
6,Dependents_Yes,-0.163398
15,OnlineBackup_Yes,-0.210712
19,TechSupport_Yes,-0.387059
7,PhoneService_Yes,-0.442177
13,OnlineSecurity_Yes,-0.467571
24,Contract_One year,-0.641291
25,Contract_Two year,-1.417141


The largest positive coefficients represent features associated with higher churn risk, while the largest negative coefficients represent features associated with lower churn risk. This helps identify which customer characteristics may be most important in predicting churn.

## Model Evaluation

We evaluate model performance using:
- Accuracy: overall correctness of predictions
- Confusion Matrix: breakdown of correct vs incorrect predictions
- Classification Report: precision, recall, and F1-score

These metrics help us understand how well the model identifies churners.

In [14]:
print("Confusion Matrix (Logistic Regression):")
print(confusion_matrix(y_test, lr_pred))

print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

Confusion Matrix (Logistic Regression):
[[934 102]
 [150 223]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409



In [16]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
0,Logistic Regression,0.821150
2,Random Forest,0.794180
1,Decision Tree,0.709723


## Model Comparison and Conclusion

The three models produced different levels of performance on the customer churn dataset. In this analysis, Logistic Regression achieved the highest accuracy, followed by Random Forest, while Decision Tree performed the weakest.

These results suggest that a simpler linear model was able to capture the churn patterns more effectively than the tree-based models used here.

This analysis can help businesses identify customers at risk of leaving and support retention strategies targeted toward those customers.